# 19 — Correct differential-abundance binomial GLM sensitivity analysis

This notebook is the repository-numbered equivalent of the historical executed notebook:

`18b_fix_DA_binomial_GLM_HVG8000_r0_6.ipynb`

The scientific/statistical code is preserved from that historical fix.

## Role in the pipeline

Notebook 18 intentionally preserves the original secondary binomial-GLM sensitivity attempt, which reproduced the historical runtime-error status pattern. This notebook is the subsequent historical fix that coerces the count/proportion columns to numeric dtypes and runs the compartment-level binomial GLM successfully.

The primary differential-abundance inference remains the sample-level non-parametric analysis from Notebook 18. This GLM remains a sensitivity analysis.

## Canonical input policy

The only input is the canonical Notebook-18 sample-level compartment composition table:

`metadata/DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv`

Hard checkpoints verify that this table contains the historical 319 compartment/sample rows, 83 samples, four conditions, and the expected per-condition sample counts before fitting the GLMs.

## Historical output checkpoints

A successful run should reproduce:

- GLM summary: `(4, 11)`
- coefficient table: `(16, 10)`
- manifest: `(3, 3)`
- all four GLM feature rows with `status == "ok"`
- all four conditions represented
- GBM as the reference condition for all coefficient rows

No H5AD is read by this notebook.

In [1]:
# =========================
# IMPORTS + PATHS
# =========================
from pathlib import Path
import os

import numpy as np
import pandas as pd
from scipy.stats import chi2


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INPUT_TSV = (
    METADATA_DIR
    / "DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv"
)

OUT_GLM = (
    METADATA_DIR
    / "DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed.tsv"
)

OUT_COEF = (
    METADATA_DIR
    / "DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed_coefficients.tsv"
)

OUT_MANIFEST = (
    METADATA_DIR
    / "DA_glm_binomial_fix_manifest_HVG8000_r0_6.tsv"
)


def write_tsv_replace(
    df,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "INPUT_TSV:",
    INPUT_TSV,
)

print(
    "exists:",
    INPUT_TSV.exists(),
)

if not INPUT_TSV.exists():
    raise FileNotFoundError(
        "Notebook 19 requires the canonical Notebook-18 sample-level "
        f"compartment table:\n{INPUT_TSV}"
    )


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_TSV: G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv
exists: True


In [2]:
# =========================
# LOAD + VALIDATE NOTEBOOK-18 INPUT
# =========================
df = pd.read_csv(
    INPUT_TSV,
    sep="\t",
)

required_columns = [
    "unit_id",
    "feature",
    "n_cells_feature",
    "n_cells_total",
    "condition",
    "dataset",
    "patient",
    "proportion",
    "unit_type",
    "feature_type",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise RuntimeError(
        "Canonical Notebook-18 input lacks required columns: "
        + ", ".join(
            missing_columns
        )
    )


# Keep final compartment-level sample summaries only.
df = df.loc[
    df[
        "feature_type"
    ]
    .astype(str)
    .eq(
        "final_compartment_major"
    )
].copy()


# Force numeric dtypes. This is the historical 18b/Notebook-19 GLM fix.
for col in [
    "n_cells_feature",
    "n_cells_total",
    "proportion",
]:
    df[
        col
    ] = pd.to_numeric(
        df[
            col
        ],
        errors="coerce",
    )


df[
    "condition"
] = df[
    "condition"
].astype(str)

df[
    "feature"
] = df[
    "feature"
].astype(str)

df[
    "unit_id"
] = df[
    "unit_id"
].astype(str)


# Remove invalid rows exactly as in the historical fix.
df = df.dropna(
    subset=[
        "n_cells_feature",
        "n_cells_total",
        "proportion",
        "condition",
    ]
)

df = df.loc[
    df[
        "n_cells_total"
    ] > 0
].copy()


print(
    "Filtered input shape:",
    df.shape,
)

display(
    df.head()
)


condition_sample_counts = (
    df.groupby(
        "condition"
    )[
        "unit_id"
    ]
    .nunique()
    .sort_index()
)


display(
    condition_sample_counts
)


EXPECTED_INPUT_SHAPE = (
    319,
    10,
)

EXPECTED_CONDITION_SAMPLE_COUNTS = {
    "GBM": 39,
    "LGG": 4,
    "ndGBM": 22,
    "rGBM": 18,
}

EXPECTED_FEATURES = {
    "cycling_malignant_like_candidate",
    "malignant_like",
    "non_malignant_or_diploid_leaning",
    "review",
}


if df.shape != EXPECTED_INPUT_SHAPE:
    raise RuntimeError(
        "Historical Notebook-19 input-shape checkpoint failed: "
        f"{df.shape} != {EXPECTED_INPUT_SHAPE}"
    )


observed_condition_sample_counts = {
    str(key): int(value)
    for key, value
    in condition_sample_counts.items()
}


if (
    observed_condition_sample_counts
    != EXPECTED_CONDITION_SAMPLE_COUNTS
):
    raise RuntimeError(
        "Historical Notebook-19 condition/sample-count checkpoint failed. "
        f"Observed: {observed_condition_sample_counts}"
    )


observed_features = set(
    df[
        "feature"
    ]
    .astype(str)
    .unique()
)


if observed_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "Historical Notebook-19 compartment-feature checkpoint failed. "
        f"Observed: {sorted(observed_features)}"
    )


if df[
    "unit_id"
].nunique() != 83:
    raise RuntimeError(
        "Historical Notebook-19 sample-count checkpoint failed: "
        f"{df['unit_id'].nunique()} != 83"
    )


print(
    "PASS: canonical Notebook-18 input matches the historical 18b input."
)


Filtered input shape: (319, 10)


,unit_id,feature,n_cells_feature,n_cells_total,condition,dataset,patient,proportion,unit_type,feature_type
0,GSM5518596_rGBM-01-A,cycling_malignant_like_candidate,46,4324,rGBM,DS3_GSE182109,01,0.010638,sample_id,final_compartment_major
1,GSM5518596_rGBM-01-A,malignant_like,123,4324,rGBM,DS3_GSE182109,01,0.028446,sample_id,final_compartment_major
2,GSM5518596_rGBM-01-A,non_malignant_or_diploid_leaning,46,4324,rGBM,DS3_GSE182109,01,0.010638,sample_id,final_compartment_major
3,GSM5518596_rGBM-01-A,review,4109,4324,rGBM,DS3_GSE182109,01,0.950278,sample_id,final_compartment_major
4,GSM5518597_rGBM-01-B,cycling_malignant_like_candidate,144,10960,rGBM,DS3_GSE182109,01,0.013139,sample_id,final_compartment_major


condition
GBM      39
LGG       4
ndGBM    22
rGBM     18
Name: unit_id, dtype: int64

PASS: canonical Notebook-18 input matches the historical 18b input.


In [3]:
# =========================
# HISTORICAL FIXED BINOMIAL GLM SENSITIVITY ANALYSIS
# =========================
# success = n cells in compartment
# total = total cells in sample
# model: proportion ~ condition
#
# This is a sensitivity check, not the main DA test.
# Main inference should still prioritize the sample-level
# non-parametric tests from Notebook 18.

import statsmodels.api as sm
import patsy


glm_rows = []
coef_rows = []


for feature, sub in df.groupby(
    "feature",
    sort=True,
):
    sub = sub.copy()

    # Need at least 2 conditions with data.
    cond_counts = (
        sub[
            "condition"
        ]
        .value_counts()
    )

    valid_conditions = (
        cond_counts
        .index
        .tolist()
    )

    if len(
        valid_conditions
    ) < 2:
        glm_rows.append(
            {
                "feature": feature,
                "test": "binomial_GLM_condition_LRT_v2",
                "pvalue": np.nan,
                "status": "skipped:<2_conditions",
                "n_units": int(
                    sub[
                        "unit_id"
                    ]
                    .nunique()
                ),
                "n_conditions": int(
                    len(
                        valid_conditions
                    )
                ),
                "ll_full": np.nan,
                "ll_null": np.nan,
                "df_diff": np.nan,
                "lrt_stat": np.nan,
            }
        )

        continue


    # Set GBM as reference if present for more interpretable coefficients.
    cond_order = [
        "GBM",
        "LGG",
        "ndGBM",
        "rGBM",
    ]

    present_order = [
        condition
        for condition in cond_order
        if condition in set(
            sub[
                "condition"
            ]
        )
    ]

    extra = [
        condition
        for condition in sorted(
            set(
                sub[
                    "condition"
                ]
            )
        )
        if condition not in present_order
    ]

    cat_order = (
        present_order
        + extra
    )

    sub[
        "condition"
    ] = pd.Categorical(
        sub[
            "condition"
        ],
        categories=cat_order,
        ordered=False,
    )


    try:
        # Endog as two-column success/failure is robust for binomial counts.
        successes = (
            sub[
                "n_cells_feature"
            ]
            .astype(float)
            .values
        )

        failures = (
            (
                sub[
                    "n_cells_total"
                ]
                - sub[
                    "n_cells_feature"
                ]
            )
            .astype(float)
            .values
        )

        y = np.column_stack(
            [
                successes,
                failures,
            ]
        )


        X_full = patsy.dmatrix(
            "C(condition)",
            sub,
            return_type="dataframe",
        )

        X_null = patsy.dmatrix(
            "1",
            sub,
            return_type="dataframe",
        )


        full_model = sm.GLM(
            y,
            X_full,
            family=sm.families.Binomial(),
        )

        null_model = sm.GLM(
            y,
            X_null,
            family=sm.families.Binomial(),
        )


        full_res = full_model.fit(
            maxiter=200
        )

        null_res = null_model.fit(
            maxiter=200
        )


        lrt_stat = 2 * (
            full_res.llf
            - null_res.llf
        )

        df_diff = int(
            full_res.df_model
            - null_res.df_model
        )

        pvalue = (
            float(
                chi2.sf(
                    lrt_stat,
                    df_diff,
                )
            )
            if df_diff > 0
            else np.nan
        )


        glm_rows.append(
            {
                "feature": feature,
                "test": "binomial_GLM_condition_LRT_v2",
                "pvalue": pvalue,
                "status": "ok",
                "n_units": int(
                    sub[
                        "unit_id"
                    ]
                    .nunique()
                ),
                "n_conditions": int(
                    len(
                        valid_conditions
                    )
                ),
                "ll_full": float(
                    full_res.llf
                ),
                "ll_null": float(
                    null_res.llf
                ),
                "df_diff": df_diff,
                "lrt_stat": float(
                    lrt_stat
                ),
            }
        )


        for term, coef, se, z, p in zip(
            full_res.params.index,
            full_res.params.values,
            full_res.bse.values,
            full_res.tvalues.values,
            full_res.pvalues.values,
        ):
            coef_rows.append(
                {
                    "feature": feature,
                    "term": term,
                    "coef_log_odds": float(
                        coef
                    ),
                    "odds_ratio": float(
                        np.exp(
                            coef
                        )
                    ),
                    "std_error": float(
                        se
                    ),
                    "z_value": float(
                        z
                    ),
                    "pvalue": float(
                        p
                    ),
                    "status": "ok",
                    "reference_condition": (
                        cat_order[
                            0
                        ]
                        if cat_order
                        else None
                    ),
                }
            )


    except Exception as error:
        glm_rows.append(
            {
                "feature": feature,
                "test": "binomial_GLM_condition_LRT_v2",
                "pvalue": np.nan,
                "status": f"error:{repr(error)}",
                "n_units": int(
                    sub[
                        "unit_id"
                    ]
                    .nunique()
                ),
                "n_conditions": int(
                    len(
                        valid_conditions
                    )
                ),
                "ll_full": np.nan,
                "ll_null": np.nan,
                "df_diff": np.nan,
                "lrt_stat": np.nan,
            }
        )


glm_df = pd.DataFrame(
    glm_rows
)

coef_df = pd.DataFrame(
    coef_rows
)


# Historical FDR correction.
try:
    from statsmodels.stats.multitest import multipletests

    ok = (
        glm_df[
            "pvalue"
        ]
        .notna()
    )

    glm_df[
        "qvalue_fdr_bh"
    ] = np.nan

    if ok.sum() > 0:
        glm_df.loc[
            ok,
            "qvalue_fdr_bh",
        ] = multipletests(
            glm_df.loc[
                ok,
                "pvalue",
            ],
            method="fdr_bh",
        )[
            1
        ]


    okc = (
        coef_df[
            "pvalue"
        ]
        .notna()
        if len(
            coef_df
        )
        else pd.Series(
            [],
            dtype=bool,
        )
    )


    if len(
        coef_df
    ):
        coef_df[
            "qvalue_fdr_bh"
        ] = np.nan

        coef_df.loc[
            okc,
            "qvalue_fdr_bh",
        ] = multipletests(
            coef_df.loc[
                okc,
                "pvalue",
            ],
            method="fdr_bh",
        )[
            1
        ]


except Exception as error:
    print(
        "[WARNING] FDR correction failed:",
        repr(
            error
        ),
    )


glm_df = (
    glm_df
    .sort_values(
        [
            "qvalue_fdr_bh",
            "pvalue",
        ],
        na_position="last",
    )
)


coef_df = (
    coef_df
    .sort_values(
        [
            "feature",
            "term",
        ]
    )
    if len(
        coef_df
    )
    else coef_df
)


# -------------------------
# Historical hard checkpoints
# -------------------------
if glm_df.shape != (
    4,
    11,
):
    raise RuntimeError(
        "Historical fixed-GLM summary checkpoint failed: "
        f"{glm_df.shape} != (4, 11)"
    )


if coef_df.shape != (
    16,
    10,
):
    raise RuntimeError(
        "Historical fixed-GLM coefficient checkpoint failed: "
        f"{coef_df.shape} != (16, 10)"
    )


if not (
    glm_df[
        "status"
    ]
    .astype(str)
    .eq(
        "ok"
    )
    .all()
):
    raise RuntimeError(
        "Historical fixed-GLM status checkpoint failed. "
        f"Observed: {glm_df['status'].tolist()}"
    )


if not (
    glm_df[
        "n_conditions"
    ]
    .astype(int)
    .eq(
        4
    )
    .all()
):
    raise RuntimeError(
        "Historical fixed-GLM condition-count checkpoint failed."
    )


if set(
    glm_df[
        "feature"
    ]
    .astype(str)
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "Historical fixed-GLM feature checkpoint failed."
    )


if not (
    coef_df[
        "reference_condition"
    ]
    .astype(str)
    .eq(
        "GBM"
    )
    .all()
):
    raise RuntimeError(
        "Historical fixed-GLM reference-condition checkpoint failed."
    )


if not (
    coef_df[
        "status"
    ]
    .astype(str)
    .eq(
        "ok"
    )
    .all()
):
    raise RuntimeError(
        "Historical fixed-GLM coefficient-status checkpoint failed."
    )


write_tsv_replace(
    glm_df,
    OUT_GLM,
)

write_tsv_replace(
    coef_df,
    OUT_COEF,
)


display(
    glm_df
)

display(
    coef_df.head(
        20
    )
)


print(
    "PASS: historical 18b / repository Notebook-19 fixed GLM reproduced successfully."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed.tsv shape=(4, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed_coefficients.tsv shape=(16, 10)


,feature,test,pvalue,status,n_units,n_conditions,ll_full,ll_null,df_diff,lrt_stat,qvalue_fdr_bh
0,cycling_malignant_like_candidate,binomial_GLM_condition_LRT_v2,0.000000e+00,ok,75,4,-17523.804324,-18524.572788,3,2001.536929,0.000000e+00
1,malignant_like,binomial_GLM_condition_LRT_v2,0.000000e+00,ok,80,4,-51857.444805,-63006.804430,3,22298.719250,0.000000e+00
3,review,binomial_GLM_condition_LRT_v2,0.000000e+00,ok,83,4,-74921.661246,-90140.846804,3,30438.371115,0.000000e+00
2,non_malignant_or_diploid_leaning,binomial_GLM_condition_LRT_v2,2.253748e-282,ok,81,4,-12626.410892,-13278.288770,3,1303.755757,2.253748e-282


,feature,term,coef_log_odds,odds_ratio,std_error,z_value,pvalue,status,reference_condition,qvalue_fdr_bh
1,cycling_malignant_like_candidate,C(condition)[T.LGG],-1.944003,0.143130,0.060629,-32.064099,1.396449e-225,ok,GBM,2.482577e-225
2,cycling_malignant_like_candidate,C(condition)[T.ndGBM],-0.175957,0.838654,0.013093,-13.438920,3.576190e-41,ok,GBM,4.401464e-41
3,cycling_malignant_like_candidate,C(condition)[T.rGBM],-0.143483,0.866335,0.013259,-10.821699,2.716942e-27,ok,GBM,3.105077e-27
0,cycling_malignant_like_candidate,Intercept,-1.898826,0.149744,0.008156,-232.799160,0.000000e+00,ok,GBM,0.000000e+00
5,malignant_like,C(condition)[T.LGG],-0.589793,0.554442,0.018853,-31.283370,7.856049e-215,ok,GBM,1.256968e-214
6,malignant_like,C(condition)[T.ndGBM],-0.786993,0.455212,0.008493,-92.658691,0.000000e+00,ok,GBM,0.000000e+00
7,malignant_like,C(condition)[T.rGBM],-1.356692,0.257511,0.009888,-137.212097,0.000000e+00,ok,GBM,0.000000e+00
4,malignant_like,Intercept,-0.022881,0.977379,0.005491,-4.167158,3.084212e-05,ok,GBM,3.084212e-05
9,non_malignant_or_diploid_leaning,C(condition)[T.LGG],0.268240,1.307660,0.030069,8.920678,4.634402e-19,ok,GBM,4.943362e-19
10,non_malignant_or_diploid_leaning,C(condition)[T.ndGBM],-0.449261,0.638099,0.017206,-26.110660,2.759046e-150,ok,GBM,3.678729e-150


PASS: historical 18b / repository Notebook-19 fixed GLM reproduced successfully.


In [4]:
# =========================
# OUTPUT MANIFEST + FINAL CHECKPOINTS
# =========================
manifest = pd.DataFrame(
    [
        {
            "path": str(
                INPUT_TSV
            ),
            "exists": (
                INPUT_TSV.exists()
            ),
            "size_bytes": (
                INPUT_TSV.stat().st_size
                if INPUT_TSV.exists()
                else np.nan
            ),
        },
        {
            "path": str(
                OUT_GLM
            ),
            "exists": (
                OUT_GLM.exists()
            ),
            "size_bytes": (
                OUT_GLM.stat().st_size
                if OUT_GLM.exists()
                else np.nan
            ),
        },
        {
            "path": str(
                OUT_COEF
            ),
            "exists": (
                OUT_COEF.exists()
            ),
            "size_bytes": (
                OUT_COEF.stat().st_size
                if OUT_COEF.exists()
                else np.nan
            ),
        },
    ]
)


write_tsv_replace(
    manifest,
    OUT_MANIFEST,
)


if manifest.shape != (
    3,
    3,
):
    raise RuntimeError(
        "Historical fixed-GLM manifest checkpoint failed: "
        f"{manifest.shape} != (3, 3)"
    )


if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more Notebook-19 GLM artifacts are missing."
    )


display(
    manifest
)


print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Historical notebook mapping:",
    "18b_fix_DA_binomial_GLM_HVG8000_r0_6.ipynb -> 19_correct_da_binomial_glm.ipynb",
)

print(
    "Input sample-level compartment table:",
    df.shape,
)

print(
    "Condition sample counts:",
    observed_condition_sample_counts,
)

print(
    "Fixed GLM summary:",
    glm_df.shape,
)

print(
    "Fixed GLM coefficients:",
    coef_df.shape,
)

print(
    "GLM statuses:",
    glm_df[
        "status"
    ]
    .value_counts()
    .to_dict(),
)

print(
    "Reference conditions:",
    coef_df[
        "reference_condition"
    ]
    .value_counts()
    .to_dict(),
)

print(
    "Manifest:",
    manifest.shape,
)

print(
    "\n[DONE] Notebook 19 corrected binomial GLM sensitivity analysis completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_fix_manifest_HVG8000_r0_6.tsv shape=(3, 3)


,path,exists,size_bytes
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,41280
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,649
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2560



FINAL CHECKPOINTS
Historical notebook mapping: 18b_fix_DA_binomial_GLM_HVG8000_r0_6.ipynb -> 19_correct_da_binomial_glm.ipynb
Input sample-level compartment table: (319, 10)
Condition sample counts: {'GBM': 39, 'LGG': 4, 'ndGBM': 22, 'rGBM': 18}
Fixed GLM summary: (4, 11)
Fixed GLM coefficients: (16, 10)
GLM statuses: {'ok': 4}
Reference conditions: {'GBM': 16}
Manifest: (3, 3)

[DONE] Notebook 19 corrected binomial GLM sensitivity analysis completed.
